# Member 05 — Data Augmentation cho hồi quy tuổi từ X-quang ngực

**Coursework:** CNN-Based Patient Age Regression from NIH ChestX-ray14.  
**Đầu vào:** grayscale `[B,1,224,224]`; **đích:** Patient Age (năm); **đầu ra:** `[B,1]`.

Notebook tuân theo `00_COURSEWORK_PLAN.md` và `01_MEMBER_TASKS.md`. Dùng nguyên split của Member 02, CNN của Member 03, so sánh E3–E1 và E4–E2. Chỉ train có augmentation ngẫu nhiên.

**Trạng thái bàn giao:** mã và notebook đã chuẩn bị; chưa huấn luyện trên NIH, chưa có ảnh before/after hay MAE thực nghiệm vì chưa có archive.zip. Kiểm thử Python/PyTorch chưa thực thi trong phiên soạn thảo. Audit metadata thực tế được lưu riêng; không dùng số liệu giả lập làm kết quả coursework.


## 1. Thiết kế thí nghiệm

| ID | Model | Loss | Train augmentation |
|---|---|---|---|
| E1 | CNN → GAP → Linear(1) | MSE | Không |
| E2 | Cùng CNN | MAE/L1 | Không |
| E3 | Cùng CNN | MSE | Có |
| E4 | Cùng CNN | MAE/L1 | Có |

Giữ seed 42, image size 224, batch size 32, Adam, learning rate 0.001, weight decay 0.0001, 10 epochs theo cấu hình trong notebook Member 01. Mỗi thí nghiệm bắt đầu từ cùng state_dict; sampler có generator riêng để augmentation không làm thay đổi thứ tự minibatch. Không early stopping/scheduler. Lưu epoch có **validation MAE nhỏ nhất**, hòa thì giữ epoch sớm hơn.

Notebook Member 04 trên main ở commit `4210d4f` dùng dữ liệu `y=3x+2+noise` và MLP, nên các số liệu đó không phải E1/E2 trên NIH. Bộ chạy này huấn luyện lại cả E1–E4 cùng recipe; không trộn kết quả hai bài toán.


## 2. Chuẩn bị môi trường và đường dẫn

Clone repository, checkout nhánh chứa phần Member 05, cài từ repository root:

```bash
python -m pip install -r CourseWork/Member_05_Augmentation/requirements.txt
```

Mở notebook trong bản clone. Giữ `train.csv`, `val.csv`, `test.csv` của Member 02 và cung cấp archive.zip có các đường dẫn `zip_member` tương ứng. Chỉnh `ZIP_PATH` bên dưới hoặc đặt biến môi trường `NIH_XRAY_ZIP`. Không commit ảnh NIH vào GitHub.

Nếu thiếu thư viện/ảnh, các cell phụ thuộc sẽ ghi rõ chưa chạy; notebook không tự tạo dữ liệu thay thế.


In [ ]:
import importlib.util
import json
import os
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "CourseWork/00_COURSEWORK_PLAN.md").is_file()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Hãy mở notebook bên trong bản clone của repository.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

MEMBER_DIR = REPO_ROOT / "CourseWork/Member_05_Augmentation"
PROCESSED_DIR = REPO_ROOT / "CourseWork/Member_02_Data/data/processed"
ZIP_PATH = Path(os.environ.get(
    "NIH_XRAY_ZIP",
    str(REPO_ROOT / "CourseWork/Member_02_Data/data/data NIH xray14/archive.zip"),
))
OUTPUT_DIR = MEMBER_DIR / "results/run_seed42"
RUN_EXPERIMENTS = True  # Có dữ liệu: chạy đủ E1–E4; dùng False nếu chỉ đọc/kiểm tra.
missing = [name for name in ("torch", "torchvision", "numpy", "pandas", "matplotlib", "PIL")
           if importlib.util.find_spec(name) is None]
DEPS_READY = not missing
DATA_READY = ZIP_PATH.is_file()
print("Repository:", REPO_ROOT)
print("Archive:", ZIP_PATH, "| exists:", DATA_READY)
print("Missing dependencies:", missing)
if not DATA_READY:
    print("CHƯA HUẤN LUYỆN: cung cấp archive.zip và chạy lại notebook.")


## 3. Xác minh split đã chốt

Chỉ đọc metadata của test để kiểm tra leakage; không mở ảnh test, không đánh giá hay chọn tham số theo test. Không chia lại dữ liệu. Audit kiểm tra cột bắt buộc, tuổi trong [1,100], bản ghi trùng, tên ảnh/zip_member, patient overlap và lưu SHA-256 của mỗi CSV.


In [ ]:
from CourseWork.Member_05_Augmentation.src.contracts import validate_splits

split_audit = validate_splits(PROCESSED_DIR)
print(json.dumps(split_audit, indent=2, ensure_ascii=False))


## 4. Baseline và augmentation

Baseline khớp Member 02: ảnh được Dataset chuyển sang mode L → Resize(224,224) → ToTensor → Normalize(mean=0.5, std=0.5).

E3/E4 thêm một RandomAffine sau Resize: xoay trong ±5°, dịch ngang/dọc tối đa 2%, nội suy bilinear, viền điền 0. Không lật ảnh, không crop mạnh, không biến dạng đàn hồi, không thay nhãn tuổi. Đây là cấu hình nhẹ được đề xuất; nhóm cần kiểm tra hình thật xem có mất vùng phổi hay tạo viền bất hợp lý trước khi chốt. Validation/test luôn chỉ có baseline, kể cả khi truyền augment=True.


In [ ]:
if DEPS_READY:
    import torch
    from PIL import Image
    from CourseWork.Member_05_Augmentation.src.transforms import build_transform
    from CourseWork.Member_02_Data.src.dataset import default_transform

    probe = Image.new("L", (256, 256), color=128)  # Chỉ kiểm tra code, không phải ảnh X-quang.
    assert torch.equal(build_transform("train")(probe), default_transform()(probe))
    for split in ("val", "test"):
        transform = build_transform(split, augment=True)
        assert torch.equal(transform(probe), transform(probe))
        assert torch.equal(transform(probe), build_transform(split, False)(probe))
    print("Baseline khớp Member 02; validation/test deterministic.")
    print("Train E3/E4:", build_transform("train", True))
else:
    print("Chưa chạy kiểm tra transform: thiếu thư viện:", missing)


## 5. Minh họa trước/sau trên ảnh TRAIN thật

Mỗi hàng là cùng một ảnh và cùng nhãn tuổi; cột đầu là baseline, ba cột sau là các biến thể. Lấy mẫu cố định seed 42 từ train. Kiểm tra hướng ảnh, trường phổi, vùng tim và mức cắt biên. Hàm lưu ảnh không làm thay đổi trạng thái RNG dùng cho training.


In [ ]:
if DEPS_READY and DATA_READY:
    from IPython.display import Image as DisplayImage, display
    from CourseWork.Member_05_Augmentation.src.reporting import save_examples

    examples_path = save_examples(
        PROCESSED_DIR / "train.csv", ZIP_PATH,
        MEMBER_DIR / "figures/augmentation_examples.png",
    )
    display(DisplayImage(filename=str(examples_path)))
else:
    print("Chưa tạo hình trên NIH: cần thư viện và archive.zip; không thay bằng ảnh giả.")


## 6. Tích hợp CNN và cấu hình

Ưu tiên `Member_03_CNN_Model/src/model.py`. Ở main đã kiểm tra, file này chưa có; `src/reference_model.py` là bản sao **nguyên văn** từ nhánh linh-brand, commit `736cbbc34de4287c8e44dea14731a6834f715ad2`, blob `f1d3cefe...`. Có cảnh báo và fingerprint khi dùng bản sao này.

Kiến trúc: 4 ConvBlock (Conv2d → BatchNorm → ReLU → MaxPool), channels 1→32→64→128→256, AdaptiveAvgPool2d(1), Flatten, Linear(256,1); 389.057 tham số. Cần nhóm trưởng/Member 03 chốt source trước lần chạy chính thức.


In [ ]:
if DEPS_READY:
    from dataclasses import asdict
    from CourseWork.Member_05_Augmentation.src.integration import model_class
    from CourseWork.Member_05_Augmentation.src.experiments import Config

    CONFIG = Config()
    model_type, model_info = model_class()
    model = model_type().eval()
    with torch.no_grad():
        output = model(torch.zeros(2, 1, 224, 224))
    assert tuple(output.shape) == (2, 1)
    print("Parameters:", sum(p.numel() for p in model.parameters()))
    print("Model source:", model_info)
    print("Config:", asdict(CONFIG))
else:
    print("Chưa kiểm tra forward pass do thiếu PyTorch.")


## 7. Chạy E1–E4

Chạy trên toàn bộ train/validation CSV, không giới hạn mẫu. Mọi ảnh được đọc trực tiếp từ ZIP qua Dataset của Member 02. Kết quả lưu train/validation loss, MAE/MSE/RMSE theo epoch; checkpoint chỉ chọn bằng validation MAE.

Thư mục output phải mới để tránh ghép baseline cũ với augmentation mới. Đổi OUTPUT_DIR khi thực hiện một lượt chạy khác. Chạy 1 epoch chỉ để kiểm tra luồng; không coi là kết quả cuối. Một lượt 10 epochs × 4 mô hình trên 78.831 ảnh train có thể cần GPU và thời gian đáng kể.


In [ ]:
run_results = None
if DEPS_READY and DATA_READY and RUN_EXPERIMENTS:
    from CourseWork.Member_05_Augmentation.src.experiments import run_matrix
    run_results = run_matrix(
        zip_path=ZIP_PATH, processed_dir=PROCESSED_DIR,
        output_dir=OUTPUT_DIR, config=CONFIG,
    )
    display(run_results)
else:
    print("Chưa chạy E1–E4. Kiểm tra thư viện, ZIP_PATH và RUN_EXPERIMENTS.")


## 8. So sánh augmentation với baseline

- MSE: E3 so với E1; MAE/L1: E4 so với E2.
- ΔMAE = MAE_augmented − MAE_baseline; âm nghĩa là tốt hơn trên validation.
- MAE và RMSE tính bằng năm; MSE tính bằng năm². Không dùng accuracy.
- Vẽ loss MSE và L1 ở các subplot riêng vì đơn vị khác nhau.


In [ ]:
if run_results is not None:
    import pandas as pd
    from IPython.display import Markdown
    display(pd.read_csv(OUTPUT_DIR / "augmentation_comparison.csv"))
    for filename in ("train_validation_loss.png", "validation_mae.png", "validation_comparison.png"):
        display(DisplayImage(filename=str(OUTPUT_DIR / "figures" / filename)))
    display(Markdown((OUTPUT_DIR / "discussion.md").read_text(encoding="utf-8")))
else:
    print("Chưa có số liệu để kết luận augmentation cải thiện hay làm giảm chất lượng.")


## 9. Thảo luận và hạn chế

Khi có kết quả, trả lời riêng cho MSE và MAE: ΔMAE là bao nhiêu, biểu đồ có dấu hiệu overfitting không, mức augmentation có che mất thông tin tuổi không? Train MAE ở E3/E4 được đo trên ảnh đã biến đổi nên không so trực tiếp khoảng cách train–val với baseline.

Không mặc định augmentation sẽ tốt hơn. Một seed chưa cho biết độ ổn định; nếu nhóm chọn nhiều seed thì cần chạy lại đầy đủ ma trận cho từng seed theo kế hoạch trước khi xem test. Tuổi dự đoán có thể chịu ảnh hưởng phân bố tuổi, tư thế chụp, chất lượng ảnh và sai lệch dữ liệu. Việc kiểm tra trực quan không thay thế đánh giá tổng quát hóa.

**Kết luận hiện tại:** metadata không rò rỉ bệnh nhân giữa ba split. Chưa có thực nghiệm NIH để kết luận hiệu quả augmentation.


## 10. Bàn giao cho Member 06 và Member 01

Bàn giao toàn bộ thư mục OUTPUT_DIR: manifest.json, E1–E4/best_model.pt, history.csv, result.json, validation_results.csv, augmentation_comparison.csv, discussion.md và các figures.

Member 06 chỉ đánh giá test sau khi đã chốt recipe/checkpoint. Lệnh đánh giá tùy chọn trong README kiểm tra fingerprint dữ liệu, source model và checkpoint rồi xuất MAE/MSE/RMSE. Notebook này không tự gọi bước test. Member 06 tiếp tục actual-vs-predicted và residual plots theo phần việc của mình.

Trước khi merge: chạy tests, chạy notebook trên NIH thật, xem hình augmentation, ghi kết quả và kiểm tra lại cấu hình chung với nhóm.


## Tài liệu đối chiếu

- [Master plan](../../00_COURSEWORK_PLAN.md)
- [Member tasks và shared contract](../../01_MEMBER_TASKS.md)
- [Member 02 Dataset](../../Member_02_Data/src/dataset.py)
- [Member 03 source đã đối chiếu](https://github.com/tanlen06-debug/UTH-Deep-Learning-nhom2/blob/736cbbc34de4287c8e44dea14731a6834f715ad2/CourseWork/Member_03_CNN_Model/src/model.py)
- [README và lệnh tái lập](../README.md)
